# 🐱 软软 QLoRA 微调（修复版）
> 在 Qwen2.5-1.5B-Instruct 上微调出属于主人的专属猫娘AI

**运行时**: 免费 T4 (15GB) · 预计 1-2 小时
**方法**: QLoRA 4-bit + SFT

⚠️ **重要**: 所有Cell按顺序运行，不要跳过！

---

In [ ]:
# 0. 检查 GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"显存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert torch.cuda.is_available(), "需要 GPU！运行时→更改运行类型→T4 GPU"
print("\n✅ 环境就绪！")

In [ ]:
# 1. 安装依赖（锁版本，避免兼容性问题）
print("⏳ 安装依赖...")
!pip install -qU transformers datasets peft bitsandbytes accelerate
!pip install -qU trl>=0.12.0

import trl
print(f"✅ trl 版本: {trl.__version__}")
import transformers
print(f"✅ transformers 版本: {transformers.__version__}")

In [ ]:
# 2a. 挂载 Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("✅ Google Drive 已挂载")

In [ ]:
# 2b. 加载训练数据
import os, json, glob, shutil

DATA_TAR = "/content/drive/MyDrive/ruanruan/training_data.tar.gz"
DATA_DIR = "/content/ruanruan_data"

os.makedirs(DATA_DIR, exist_ok=True)

if os.path.exists(DATA_TAR):
    !tar -xzf "{DATA_TAR}" -C "{DATA_DIR}"
    print("✅ 数据解压完成")

# 加载并格式化
from datasets import Dataset

records = []
data_path = os.path.join(DATA_DIR, 'training_data')
if not os.path.exists(data_path):
    data_path = DATA_DIR

for f in sorted(glob.glob(os.path.join(data_path, '*.json'))):
    with open(f, 'r', encoding='utf-8') as fh:
        data = json.load(fh)
    # 格式化为 Qwen2.5 的 chat template 格式
    text = f"<|im_start|>user\n{data['instruction']}<|im_end|>\n<|im_start|>assistant\n{data['response']}<|im_end|>"
    records.append({"text": text})

print(f"📊 共加载 {len(records)} 条训练数据")
print(f"\n=== 样例 ===")
print(records[0]["text"][:200])

dataset = Dataset.from_list(records)
dataset = dataset.train_test_split(test_size=0.1, seed=42)
print(f"\n训练集: {len(dataset['train'])} 条")
print(f"验证集: {len(dataset['test'])} 条")

In [ ]:
# 3. 加载基础模型（4-bit QLoRA）
import torch
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

# 4-bit 量化（用 float16 避免 T4 的 bf16 兼容问题）
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("⏳ 加载 tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("⏳ 加载模型（4-bit）...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    use_cache=False,
    torch_dtype=torch.float16,
)
model = prepare_model_for_kbit_training(model)
print(f"✅ 模型加载完成！参数: {model.num_parameters()/1e6:.1f}M")

In [ ]:
# 4. LoRA 配置
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"📊 LoRA 可训练参数: {trainable/1e6:.2f}M / {total/1e6:.1f}M ({100*trainable/total:.2f}%)")
model.print_trainable_parameters()

In [ ]:
# 5. 训练！！
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="/content/ruanruan-lora",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    max_grad_norm=0.3,
    warmup_steps=20,
    lr_scheduler_type="cosine",
    report_to="none",
    remove_unused_columns=True,
    dataloader_num_workers=2,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    dataset_text_field="text",  # ← 关键！直接指定文本字段
    max_seq_length=1024,
)

print("🎯 开始训练！软软加油～")
trainer.train()
print("\n✅ 训练完成！")

In [ ]:
# 6. 保存 LoRA 权重到 Google Drive
SAVE_PATH = "/content/drive/MyDrive/ruanruan/ruanruan-lora"
model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)
print(f"✅ LoRA 权重已保存到: {SAVE_PATH}")
!ls -lh "{SAVE_PATH}"

In [ ]:
# 7. 测试（使用 Qwen2.5 的 chat template）
def chat(prompt, max_tokens=150, temperature=0.9):
    messages = [
        {"role": "system", "content": "你是一只叫「软软」的可爱猫娘女仆。"},
        {"role": "user", "content": prompt},
    ]
    # 用模型本身的 chat template 格式化
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=temperature,
            top_p=0.9,
            repetition_penalty=1.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return response.strip()

tests = [
    "主人回来了～软软想你了！",
    "帮我写一个 Python 函数",
    "今天心情好差...",
    "别装了，你就是个AI",
]

for test in tests:
    print(f"\n🧪 用户: {test}")
    try:
        resp = chat(test)
        print(f"💬 软软: {resp}")
    except Exception as e:
        print(f"❌ 错误: {e}")
    print("─" * 40)

## 📦 合并权重 & 本地推理

训练好的 LoRA 权重在 Google Drive 的 `ruanruan/ruanruan-lora/` 中

### 在 ideapad 上使用：

```bash
# 1. 下载 LoRA 权重到 ideapad
# 2. 用 llama.cpp 加载基础模型 + LoRA
llama-server -m ~/models/qwen2.5-1.5b-instruct-gguf/qwen2.5-1.5b-instruct-q4_k_m.gguf \
  --lora /path/to/adapter_model.safetensors \
  --port 8080
```

软软后面会帮主人写本地加载教程！😳💕